# Transformers from scratch: companion notebook
This notebook re-computes **every number** printed in `Transformer_Guide.pdf`, in the same order as the guide, and checks them with `assert`.

* Hand-worked parts (Parts 2 to 8) use float64 and weights rounded to 1 decimal so the hand arithmetic is exact.
* Part 9 trains the tiny GPT for real (seed 0, Adam lr 0.01, 300 steps, CPU, float32).
* The last cell compares everything with `src/numbers.json`, the file the PDF was built from.

## 0 · Setup
Vocabulary, sentence and ids: identical to the RNN notes.

In [1]:
import math, torch, torch.nn as nn, torch.nn.functional as F
torch.set_printoptions(precision=4, sci_mode=False)
NUM = {}                                   # every number the guide prints is stored here
vocab = ['a', 'bad', 'monkey', 'is', 'still']
stoi = {w: i for i, w in enumerate(vocab)}  # word -> id
sentence = "a bad monkey is still a monkey".split()
ids = [stoi[w] for w in sentence]
print(ids)

[0, 1, 2, 3, 4, 0, 2]


In [2]:
assert ids == [0, 1, 2, 3, 4, 0, 2]
NUM['ids'] = ids; NUM['vocab'] = vocab
r1 = lambda t: torch.round(t * 10) / 10    # round to 1 decimal (for hand-friendly numbers)

## 1 · Why RNNs fall short
Hand example: a signal multiplied by 0.5 at every step.

In [3]:
fade = [0.5 ** k for k in range(11)]
NUM['fade'] = fade
assert abs(fade[10] - 0.0009765625) < 1e-12

Measured: gradient of the last hidden state w.r.t. each earlier input in an untrained RNN.

In [4]:
torch.manual_seed(0)
rnn = nn.RNN(input_size=4, hidden_size=8, batch_first=True)
x = torch.randn(1, 20, 4, requires_grad=True)   # 20 random input vectors
out, h_last = rnn(x)
h_last.sum().backward()                         # how much does h_20 depend on each x_t?
g = x.grad[0].norm(dim=1)                       # one gradient size per step
rel = (g / g[-1]).flip(0)                       # steps back: 0, 1, 2, ...
print(rel[[0, 1, 2, 5, 10, 19]])

tensor([1.0000, 0.3358, 0.0647, 0.0072, 0.0002, 0.0000])


In [5]:
NUM['rnn_rel'] = rel.tolist()
assert rel[0] == 1 and rel[10] < 0.05

## 2 · Tokens, embeddings, positions
Seed-0 embedding table (same seed as the RNN notes), rounded to 1 decimal.

In [6]:
torch.manual_seed(0)
E = r1(torch.randn(5, 4).double())        # 5 words x d_model=4
print(E)

tensor([[-1.1000, -1.2000, -0.3000, -0.4000],
        [ 0.6000, -1.6000, -0.3000,  1.9000],
        [ 0.5000, -0.2000,  1.4000,  0.3000],
        [ 1.4000,  1.6000,  0.9000, -0.8000],
        [ 0.9000,  1.3000,  2.0000,  0.1000]], dtype=torch.float64)


In [7]:
NUM['E'] = E.tolist()

Sinusoidal positional encoding.

In [8]:
def sinusoidal_pe(T, d):
    pos = torch.arange(T, dtype=torch.float64).unsqueeze(1)  # (T,1): 0,1,2,...
    two_i = torch.arange(0, d, 2, dtype=torch.float64)       # 0,2,4,... (= 2i)
    div = 10000 ** (two_i / d)                               # 10000^(2i/d)
    pe = torch.zeros(T, d, dtype=torch.float64)
    pe[:, 0::2] = torch.sin(pos / div)                       # even columns
    pe[:, 1::2] = torch.cos(pos / div)                       # odd columns
    return pe
PE = sinusoidal_pe(3, 4)
print(PE)

tensor([[ 0.0000,  1.0000,  0.0000,  1.0000],
        [ 0.8415,  0.5403,  0.0100,  1.0000],
        [ 0.9093, -0.4161,  0.0200,  0.9998]], dtype=torch.float64)


In [9]:
assert torch.allclose(PE[1], torch.tensor([math.sin(1), math.cos(1), math.sin(0.01), math.cos(0.01)], dtype=torch.float64))
NUM['PE3'] = PE.tolist()
PEbig = sinusoidal_pe(50, 16); NUM['PEbig'] = PEbig.tolist()
# dot products between positions (similarity falls with distance)
NUM['pe_dot'] = [float(PEbig[10] @ PEbig[10 + k]) for k in range(0, 21)]

Without positions, attention is order-blind: shuffling the words only shuffles the outputs.

In [10]:
torch.manual_seed(1)
Wq_, Wk_, Wv_ = [r1((torch.randn(4, 4) * 0.5).double()) for _ in range(3)]   # same seed-1 weights as Part 3
def attend(Z):
    return torch.softmax((Z @ Wq_) @ (Z @ Wk_).T / 2, -1) @ (Z @ Wv_)
Xe = E[[0, 1, 2]]                     # "a bad monkey", embeddings only (no positions)
perm = [2, 0, 1]                      # "monkey a bad"
assert torch.allclose(attend(Xe[perm]), attend(Xe)[perm])
NUM['perm_O'] = attend(Xe).tolist(); NUM['perm_Operm'] = attend(Xe[perm]).tolist()

X = embedding + position, rounded to 1 decimal for hand work. Sentence "a bad monkey".

In [11]:
hand_ids = [0, 1, 2]                    # "a bad monkey"
X = r1(E[hand_ids] + PE)                # (3, 4) rounded so you can follow by hand
print(X)

tensor([[-1.1000, -0.2000, -0.3000,  0.6000],
        [ 1.4000, -1.1000, -0.3000,  2.9000],
        [ 1.4000, -0.6000,  1.4000,  1.3000]], dtype=torch.float64)


In [12]:
NUM['Ex'] = E[hand_ids].tolist(); NUM['ExPE'] = (E[hand_ids] + PE).tolist(); NUM['X'] = X.tolist()

## 3 · Self-attention by hand
Seeded weights (seed 1), scaled by 0.5 and rounded to 1 decimal.

In [13]:
torch.manual_seed(1)
W_Q, W_K, W_V, W_O = [r1((torch.randn(4, 4) * 0.5).double()) for _ in range(4)]

In [14]:
Q = X @ W_Q          # (3,4) @ (4,4) -> (3,4)
K = X @ W_K
V = X @ W_V
S = Q @ K.T          # raw scores (3,3)
d_k = 4
S_scaled = S / math.sqrt(d_k)
A = torch.softmax(S_scaled, dim=-1)    # each row sums to 1
O = A @ V            # weighted sum of value vectors
print(A)
print(O)

tensor([[0.7574, 0.1312, 0.1114],
        [0.7939, 0.1457, 0.0604],
        [0.3743, 0.3599, 0.2658]], dtype=torch.float64)
tensor([[ 0.7062, -0.5009,  0.7092, -0.0239],
        [ 0.6309, -0.5085,  0.7077,  0.0117],
        [ 2.2760, -0.7917,  1.2592, -0.5437]], dtype=torch.float64)


In [15]:
for k, v in dict(W_Q=W_Q, W_K=W_K, W_V=W_V, W_O=W_O, Q=Q, K=K, V=V, S=S, S_scaled=S_scaled, A=A, O=O).items(): NUM[k] = v.tolist()
# hand softmax, row by row
ex = torch.exp(S_scaled); NUM['expS'] = ex.tolist(); NUM['expS_sum'] = ex.sum(1).tolist()
assert torch.allclose(ex / ex.sum(1, keepdim=True), A)
# explicit loops == matrix products
Qh = torch.tensor([[sum(X[t, j] * W_Q[j, c] for j in range(4)) for c in range(4)] for t in range(3)], dtype=torch.float64)
assert torch.allclose(Qh, Q)
assert torch.allclose(A.sum(1), torch.ones(3, dtype=torch.float64))
# PyTorch's built-in attention gives the same answer
assert torch.allclose(F.scaled_dot_product_attention(Q[None], K[None], V[None])[0], O)

Why divide by sqrt(d_k): measured spread of dot products of random unit-variance vectors.

In [16]:
g = torch.Generator().manual_seed(0)
for d in [4, 64, 512]:
    q = torch.randn(100000, d, generator=g)
    k = torch.randn(100000, d, generator=g)
    dots = (q * k).sum(1)                  # 100 000 dot products
    print(d, round(dots.std().item(), 2),
          round((dots / math.sqrt(d)).std().item(), 2))

4 2.0 1.0
64 8.01 1.0


512 22.68 1.0


In [17]:
g = torch.Generator().manual_seed(0); NUM['sqrt_demo'] = []
for d in [4, 64, 512]:
    q = torch.randn(100000, d, generator=g); k = torch.randn(100000, d, generator=g); dots = (q * k).sum(1)
    NUM['sqrt_demo'].append([d, dots.std().item(), (dots / math.sqrt(d)).std().item()])
z = torch.tensor([2.0, 1.0, 0.0], dtype=torch.float64)
NUM['soft_small'] = torch.softmax(z, 0).tolist(); NUM['soft_big'] = torch.softmax(z * 8, 0).tolist()
# gradient of softmax output 0 w.r.t. its input: tiny when saturated
def sm_grad(zz):
    zz = zz.clone().requires_grad_(True); torch.softmax(zz, 0)[0].backward(); return zz.grad.abs().max().item()
NUM['soft_grad_small'] = sm_grad(z); NUM['soft_grad_big'] = sm_grad(z * 8)

## 4 · Causal mask

In [18]:
T = 3
ones = torch.ones(T, T, dtype=torch.bool)
mask = torch.triu(ones, diagonal=1)    # True above the diagonal
S_masked = S_scaled.masked_fill(mask, float('-inf'))
A_causal = torch.softmax(S_masked, dim=-1)
O_causal = A_causal @ V
print(A_causal)

tensor([[1.0000, 0.0000, 0.0000],
        [0.8449, 0.1551, 0.0000],
        [0.3743, 0.3599, 0.2658]], dtype=torch.float64)


In [19]:
NUM['mask'] = mask.int().tolist(); NUM['S_masked'] = [[None if math.isinf(v) else v for v in r] for r in S_masked.tolist()]
NUM['A_causal'] = A_causal.tolist(); NUM['O_causal'] = O_causal.tolist()
NUM['expS_causal_row1'] = [math.exp(S_scaled[1, 0].item()), math.exp(S_scaled[1, 1].item())]
assert torch.allclose(F.scaled_dot_product_attention(Q[None], K[None], V[None], is_causal=True)[0], O_causal)
assert A_causal[0, 0] == 1 and (A_causal.triu(1) == 0).all()

## 5 · Multi-head attention (2 heads of size 2, causal)

In [20]:
h, d_head = 2, 2
heads = []
for i in range(h):
    cols = slice(i * d_head, (i + 1) * d_head)          # head 0: cols 0-1, head 1: cols 2-3
    s = Q[:, cols] @ K[:, cols].T / math.sqrt(d_head)
    s = s.masked_fill(mask, float('-inf'))
    a = torch.softmax(s, dim=-1)
    heads.append(a @ V[:, cols])                        # (3, 2)
concat = torch.cat(heads, dim=-1)                       # (3, 4)
mha_out = concat @ W_O
print(mha_out)

tensor([[ 0.2760, -0.0740, -0.7460, -0.5120],
        [-0.4702,  0.8165,  0.3472, -0.6964],
        [-1.7985,  1.3238,  0.5355, -1.3264]], dtype=torch.float64)


In [21]:
NUM['heads'] = []
for i in range(h):
    cols = slice(i * d_head, (i + 1) * d_head)
    s = Q[:, cols] @ K[:, cols].T; ss = s / math.sqrt(d_head)
    a = torch.softmax(ss.masked_fill(mask, float('-inf')), -1)
    NUM['heads'].append(dict(Q=Q[:, cols].tolist(), K=K[:, cols].tolist(), V=V[:, cols].tolist(), S=s.tolist(), Ss=ss.tolist(), A=a.tolist(), O=(a @ V[:, cols]).tolist(),
                             exp_row2=[math.exp(v) for v in ss[2].tolist()]))
NUM['concat'] = concat.tolist(); NUM['mha_out'] = mha_out.tolist()
# check against PyTorch's own nn.MultiheadAttention with the same weights
mha = nn.MultiheadAttention(4, 2, bias=False, batch_first=True).double()
with torch.no_grad():
    mha.in_proj_weight.copy_(torch.cat([W_Q.T, W_K.T, W_V.T]))   # nn stores W transposed
    mha.out_proj.weight.copy_(W_O.T)
ref, _ = mha(X[None], X[None], X[None], attn_mask=mask)
assert torch.allclose(ref[0], mha_out)
NUM['mha_check'] = (ref[0] - mha_out).abs().max().item()

## 6 · Residual, LayerNorm, FFN (worked on the token "monkey", row 2)

In [22]:
r = X + mha_out                       # residual: add the input back
x2 = r[2]                             # the "monkey" row
mu = x2.mean()
var = ((x2 - mu) ** 2).mean()         # biased variance (divide by d, not d-1)
x_hat = (x2 - mu) / torch.sqrt(var + 1e-5)
print(mu, var, x_hat, sep='\n')

tensor(0.5586, dtype=torch.float64)
tensor(0.7953, dtype=torch.float64)
tensor([-1.0732,  0.1853,  1.5439, -0.6559], dtype=torch.float64)


In [23]:
ln = nn.LayerNorm(4).double()          # gamma = 1, beta = 0 at initialisation
with torch.no_grad(): ln_ref = ln(r)
assert torch.allclose(ln_ref[2], x_hat)
NUM['r'] = r.tolist(); NUM['ln_mu'] = mu.item(); NUM['ln_var'] = var.item(); NUM['ln_std'] = math.sqrt(var.item() + 1e-5)
NUM['x_hat'] = x_hat.tolist(); NUM['x_hat_mean'] = x_hat.mean().item(); NUM['x_hat_var'] = (x_hat ** 2).mean().item()
NUM['ln_r'] = ln_ref.tolist()

Feed-forward network 4 -> 8 -> 4 with ReLU (seed 2 weights, 1 decimal).

In [24]:
torch.manual_seed(2)
W1 = r1((torch.randn(4, 8) * 0.5).double())
b1 = r1((torch.randn(8) * 0.5).double())
W2 = r1((torch.randn(8, 4) * 0.5).double())
b2 = r1((torch.randn(4) * 0.5).double())
pre = x_hat @ W1 + b1                 # (8,) before ReLU
hid = torch.relu(pre)                 # negatives -> 0
ffn_out = hid @ W2 + b2               # (4,)
out2 = r[2] + ffn_out                 # second residual
print(pre, hid, ffn_out, out2, sep='\n')

tensor([-0.6223, -0.7771,  0.0125, -0.0298,  0.6710, -1.9116,  1.3588, -0.6025],
       dtype=torch.float64)
tensor([0.0000, 0.0000, 0.0125, 0.0000, 0.6710, 0.0000, 1.3588, 0.0000],
       dtype=torch.float64)
tensor([ 1.8023, -0.7882, -0.4375, -0.9055], dtype=torch.float64)
tensor([ 1.4038, -0.0644,  1.4979, -0.9319], dtype=torch.float64)


In [25]:
for k, v in dict(W1=W1, b1=b1, W2=W2, b2=b2, ffn_pre=pre, ffn_hid=hid, ffn_out=ffn_out, out2=out2).items(): NUM[k] = v.tolist()
ffn = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 4)).double()
with torch.no_grad():
    ffn[0].weight.copy_(W1.T); ffn[0].bias.copy_(b1); ffn[2].weight.copy_(W2.T); ffn[2].bias.copy_(b2)
    assert torch.allclose(ffn(x_hat), ffn_out)

## 7 · One full pre-LN block with the same hand weights

In [26]:
def layer_norm(x, eps=1e-5):
    mu = x.mean(-1, keepdim=True)
    var = ((x - mu) ** 2).mean(-1, keepdim=True)
    return (x - mu) / torch.sqrt(var + eps)
def causal_mha(x):
    q, k, v = x @ W_Q, x @ W_K, x @ W_V
    outs = []
    for i in range(2):
        c = slice(2 * i, 2 * i + 2)
        s = q[:, c] @ k[:, c].T / math.sqrt(2)
        s = s.masked_fill(mask, float('-inf'))
        outs.append(torch.softmax(s, -1) @ v[:, c])
    return torch.cat(outs, -1) @ W_O
def block(x):
    x = x + causal_mha(layer_norm(x))                         # sub-layer 1
    x = x + torch.relu(layer_norm(x) @ W1 + b1) @ W2 + b2     # sub-layer 2
    return x
Y1 = block(X)
print(Y1)

tensor([[-0.7460, -0.3420, -3.0097, -1.0473],
        [-2.0080,  0.9267, -2.7395,  3.0564],
        [ 1.5231, -0.2426,  1.2268, -0.5166]], dtype=torch.float64)


In [27]:
L1 = layer_norm(X); a1 = causal_mha(L1); R1 = X + a1; L2 = layer_norm(R1); f1 = torch.relu(L2 @ W1 + b1) @ W2 + b2
NUM['blk'] = dict(L1=L1.tolist(), a1=a1.tolist(), R1=R1.tolist(), L2=L2.tolist(), f1=f1.tolist(), Y1=Y1.tolist())
Y2 = block(Y1); NUM['blk']['Y2'] = Y2.tolist()          # same block applied twice = shape stays (3,4)
assert Y1.shape == X.shape == Y2.shape

## 8 · Output layer, softmax, cross-entropy, p - y (untrained, seed 3)

In [28]:
torch.manual_seed(3)
W_vocab = r1((torch.randn(4, 5) * 0.5).double())   # d_model=4 -> V=5
hf = layer_norm(Y1)[2]                 # final LayerNorm, token "monkey"
z = hf @ W_vocab                       # 5 logits
p = torch.softmax(z, dim=-1)
target = stoi['is']                    # after "a bad monkey" comes "is"
loss = -torch.log(p[target])
grad = p - F.one_hot(torch.tensor(target), 5)
print(z, p, loss, grad, sep='\n')

tensor([ 0.1924,  0.0858, -0.1655, -0.2307,  0.5126], dtype=torch.float64)
tensor([0.2160, 0.1941, 0.1510, 0.1415, 0.2975], dtype=torch.float64)
tensor(1.9558, dtype=torch.float64)
tensor([ 0.2160,  0.1941,  0.1510, -0.8585,  0.2975], dtype=torch.float64)


In [29]:
NUM['W_vocab'] = W_vocab.tolist(); NUM['hf'] = hf.tolist(); NUM['z'] = z.tolist(); NUM['p'] = p.tolist(); NUM['loss_hand'] = loss.item()
NUM['grad'] = grad.tolist(); NUM['ln5'] = math.log(5); NUM['expz'] = torch.exp(z).tolist()
zz = z.clone().requires_grad_(True); F.cross_entropy(zz[None], torch.tensor([target])).backward()
assert torch.allclose(zz.grad, grad.double())
assert abs(F.cross_entropy(z[None], torch.tensor([target])).item() - loss.item()) < 1e-12

## 9 · A tiny decoder-only transformer, trained for real

In [30]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_head):
        super().__init__()
        self.h, self.dk = n_head, d_model // n_head
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)
    def make_mask(self, T):           # True = "not allowed to look"
        ones = torch.ones(T, T, dtype=torch.bool)
        return torch.triu(ones, diagonal=1)
    def forward(self, x):             # x: (B, T, d_model)
        B, T, D = x.shape
        def split(t):                 # (B,T,D) -> (B,h,T,dk)
            return t.view(B, T, self.h, -1).transpose(1, 2)
        q = split(self.W_Q(x))
        k = split(self.W_K(x))
        v = split(self.W_V(x))
        s = q @ k.transpose(-2, -1) / math.sqrt(self.dk)   # (B, h, T, T)
        s = s.masked_fill(self.make_mask(T), float('-inf'))
        self.att = torch.softmax(s, dim=-1)                # saved for heat maps
        out = self.att @ v                   # (B, h, T, dk)
        out = out.transpose(1, 2).reshape(B, T, D)  # concat
        return self.W_O(out)

In [31]:
class Block(nn.Module):
    def __init__(self, d_model, n_head, d_ff):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_head)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.ReLU(),
            nn.Linear(d_ff, d_model))
    def forward(self, x):
        x = x + self.attn(self.ln1(x))     # look at earlier words
        x = x + self.ffn(self.ln2(x))      # think about each word alone
        return x

class TinyGPT(nn.Module):
    def __init__(self, V, d_model, n_head, d_ff,
                 n_layer, T_max):
        super().__init__()
        self.tok = nn.Embedding(V, d_model)
        pe = sinusoidal_pe(T_max, d_model).float()
        self.register_buffer('pe', pe)     # fixed, not learned
        blocks = [Block(d_model, n_head, d_ff)
                  for _ in range(n_layer)]
        self.blocks = nn.ModuleList(blocks)
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, V)
    def forward(self, idx):                # idx: (B, T) word ids
        x = self.tok(idx) + self.pe[:idx.shape[1]]
        for blk in self.blocks:
            x = blk(x)
        return self.head(self.ln_f(x))     # (B, T, V) logits

In [32]:
torch.manual_seed(0)
model = TinyGPT(V=5, d_model=16, n_head=2, d_ff=64,
                n_layer=1, T_max=6)
for name, p_ in model.named_parameters():
    shape = str(tuple(p_.shape))
    print(f'{name:25s} {shape:9s} {p_.numel():>5}')
total = sum(p_.numel() for p_ in model.parameters())
print(f"{'total':35s} {total:>5}")

tok.weight                (5, 16)      80
blocks.0.ln1.weight       (16,)        16
blocks.0.ln1.bias         (16,)        16
blocks.0.attn.W_Q.weight  (16, 16)    256
blocks.0.attn.W_K.weight  (16, 16)    256
blocks.0.attn.W_V.weight  (16, 16)    256
blocks.0.attn.W_O.weight  (16, 16)    256
blocks.0.ln2.weight       (16,)        16
blocks.0.ln2.bias         (16,)        16
blocks.0.ffn.0.weight     (64, 16)   1024
blocks.0.ffn.0.bias       (64,)        64
blocks.0.ffn.2.weight     (16, 64)   1024
blocks.0.ffn.2.bias       (16,)        16
ln_f.weight               (16,)        16
ln_f.bias                 (16,)        16
head.weight               (5, 16)      80
head.bias                 (5,)          5
total                                3413


In [33]:
NUM['params'] = [[n_, list(p_.shape), p_.numel()] for n_, p_ in model.named_parameters()]
NUM['n_params'] = sum(p_.numel() for p_ in model.parameters())
assert NUM['n_params'] == sum(x_[2] for x_ in NUM['params'])

In [34]:
x = torch.tensor([ids[:-1]])     # a bad monkey is still a
y = torch.tensor([ids[1:]])      # bad monkey is still a monkey
opt = torch.optim.Adam(model.parameters(), lr=0.01)
losses = []
for step in range(300):
    logits = model(x)                                  # (1, 6, 5)
    loss = F.cross_entropy(logits.view(-1, 5), y.view(-1))
    opt.zero_grad()
    loss.backward()
    opt.step()
    losses.append(loss.item())
print(round(losses[0], 4), round(losses[99], 4),
      round(losses[-1], 6))

2.2412 0.0025 0.000537


In [35]:
# untrained predictions: rebuild the identical seed-0 model (training changed `model`)
torch.manual_seed(0)
model0 = TinyGPT(V=5, d_model=16, n_head=2, d_ff=64, n_layer=1, T_max=6)
with torch.no_grad():
    p0 = torch.softmax(model0(x)[0], -1); att0 = [model0.blocks[0].attn.att[0, i].tolist() for i in range(2)]
    p1 = torch.softmax(model(x)[0], -1); att1 = [model.blocks[0].attn.att[0, i].tolist() for i in range(2)]
    final_loss = F.cross_entropy(model(x).view(-1, 5), y.view(-1)).item()
NUM['losses'] = losses; NUM['p_untrained'] = p0.tolist(); NUM['p_trained'] = p1.tolist()
NUM['att_untrained'] = att0; NUM['att_trained'] = att1; NUM['final_loss'] = final_loss
NUM['loss0'] = losses[0]
NUM['acc_untrained'] = int((p0.argmax(-1) == y[0]).sum()); NUM['acc_trained'] = int((p1.argmax(-1) == y[0]).sum())
NUM['step_first_6of6'] = None
torch.manual_seed(0); mtmp = TinyGPT(5, 16, 2, 64, 1, 6); otmp = torch.optim.Adam(mtmp.parameters(), lr=0.01)
acc_steps = []
for st in range(300):
    lg = mtmp(x)
    acc_steps.append(int((lg[0].argmax(-1) == y[0]).sum()))
    if NUM['step_first_6of6'] is None and acc_steps[-1] == 6: NUM['step_first_6of6'] = st
    l_ = F.cross_entropy(lg.view(-1, 5), y.view(-1)); otmp.zero_grad(); l_.backward(); otmp.step()
assert abs(l_.item() - losses[-1]) < 1e-6   # same seed -> same run (determinism check)
assert NUM['acc_trained'] == 6
assert all(a_ == 6 for a_ in acc_steps[NUM['step_first_6of6']:])   # stays 6/6 once reached
NUM['acc_steps'] = acc_steps
print(p1[0], p1[5])

tensor([0.0002, 0.9994, 0.0002, 0.0002, 0.0000]) tensor([0.0001, 0.0001, 0.9997, 0.0001, 0.0001])


Ablation: which ingredient separates the two "a"s?

In [36]:
class SelfOnlyAttention(CausalSelfAttention):
    def make_mask(self, T):          # each word may look ONLY at itself
        return ~torch.eye(T, dtype=torch.bool)
def run(use_pe=True, self_only=False, seed=0, steps=300):
    torch.manual_seed(seed)
    m = TinyGPT(5, 16, 2, 64, 1, 6)
    if self_only:
        old = m.blocks[0].attn; new = SelfOnlyAttention(16, 2); new.load_state_dict(old.state_dict()); m.blocks[0].attn = new
    if not use_pe: m.pe.zero_()
    o = torch.optim.Adam(m.parameters(), lr=0.01)
    for _ in range(steps):
        l = F.cross_entropy(m(x).view(-1, 5), y.view(-1)); o.zero_grad(); l.backward(); o.step()
    with torch.no_grad():
        pp = torch.softmax(m(x)[0], -1)
        return dict(loss=F.cross_entropy(m(x).view(-1, 5), y.view(-1)).item(), p0=pp[0].tolist(), p5=pp[5].tolist(),
                    acc=int((pp.argmax(-1) == y[0]).sum()))
NUM['ablate'] = {'full': run(), 'no_pe': run(use_pe=False), 'self_only': run(self_only=True), 'self_only_no_pe': run(use_pe=False, self_only=True)}
NUM['floor'] = 2 * math.log(2) / 6
for k, v in NUM['ablate'].items(): print(k, round(v['loss'], 4), v['acc'], [round(t, 3) for t in v['p5']])
assert abs(NUM['ablate']['full']['loss'] - NUM['final_loss']) < 1e-6

full 0.0005 6 [0.0, 0.0, 1.0, 0.0, 0.0]
no_pe 0.0005 6 [0.0, 0.0, 1.0, 0.0, 0.0]
self_only 0.0005 6 [0.0, 0.0, 1.0, 0.0, 0.0]
self_only_no_pe 0.2316 5 [0.0, 0.5, 0.5, 0.0, 0.0]


## 10 · Generation

In [37]:
@torch.no_grad()
def generate(model, start, n_new, temperature=1.0,
             greedy=True, gen=None):
    idx = torch.tensor([[stoi[w] for w in start]])
    for _ in range(n_new):
        logits = model(idx[:, -6:])[0, -1]           # last position only
        p = torch.softmax(logits / temperature, dim=-1)
        if greedy:
            nxt = p.argmax()
        else:
            nxt = torch.multinomial(p, 1, generator=gen)[0]
        idx = torch.cat([idx, nxt.view(1, 1)], dim=1)
    return ' '.join(vocab[i] for i in idx[0])
print(generate(model, ['a'], 6))

a bad monkey is still a monkey


In [38]:
NUM['greedy'] = generate(model, ['a'], 6)
NUM['greedy_monkey'] = generate(model, ['monkey'], 5)
with torch.no_grad():
    ctx = torch.tensor([ids[:6]]); zlast = model(ctx)[0, -1]
NUM['z_last'] = zlast.tolist()
NUM['temp'] = {str(T_): torch.softmax(zlast / T_, -1).tolist() for T_ in [0.5, 1.0, 2.0, 5.0]}
# sampling: 5 samples per temperature, fixed generator seed
NUM['samples'] = {}
for T_ in [1.0, 2.0, 5.0]:
    g = torch.Generator().manual_seed(0)
    NUM['samples'][str(T_)] = [generate(model, ['a'], 6, T_, greedy=False, gen=g) for _ in range(5)]
# step-by-step greedy table
steps_ = []; cur = ['a']
with torch.no_grad():
    for _ in range(6):
        pr = torch.softmax(model(torch.tensor([[stoi[w] for w in cur]]))[0, -1], -1)
        k = int(pr.argmax()); steps_.append([' '.join(cur), vocab[k], pr[k].item()]); cur = cur + [vocab[k]]
NUM['greedy_steps'] = steps_
print(NUM['samples'])

{'1.0': ['a bad monkey is still a monkey', 'a bad monkey is still a monkey', 'a bad monkey is still a monkey', 'a bad monkey is still a monkey', 'a bad monkey is still a monkey'], '2.0': ['a bad monkey is still a monkey', 'a bad monkey is still a is', 'a bad monkey still a monkey is', 'a bad monkey is still a monkey', 'a bad monkey is still a monkey'], '5.0': ['a bad monkey is monkey monkey still', 'a is still bad monkey monkey is', 'a bad is still a monkey still', 'a is a a still is a', 'a monkey is still a still a']}


## 11 · KV cache sanity check: the last row of attention only needs the cached K and V

In [39]:
with torch.no_grad():
    att = model.blocks[0].attn
    h_in = model.blocks[0].ln1(model.tok(x) + model.pe[:6])
    full = att(h_in)[0, -1]                       # recompute everything
    q = att.W_Q(h_in[:, -1:]); k = att.W_K(h_in); v = att.W_V(h_in)   # k, v = the cache
    q = q.view(1, 1, 2, 8).transpose(1, 2); k = k.view(1, 6, 2, 8).transpose(1, 2); v = v.view(1, 6, 2, 8).transpose(1, 2)
    a = torch.softmax(q @ k.transpose(-2, -1) / math.sqrt(8), -1)
    one = att.W_O((a @ v).transpose(1, 2).reshape(1, 1, 16))[0, 0]
NUM['kv_diff'] = (full - one).abs().max().item()
assert NUM['kv_diff'] < 1e-5

## 12 · Numbers used in the practice answers and Part 11 arithmetic

In [40]:
xq = torch.tensor([1., 2., 3., 4.])
NUM['q_ln'] = ((xq - xq.mean()) / xq.var(unbiased=False).sqrt()).tolist()
NUM['q_softmax_000'] = torch.softmax(torch.zeros(3), 0).tolist()
NUM['q_masked_T4'] = int(torch.triu(torch.ones(4, 4, dtype=torch.bool), 1).sum())
NUM['q_S_aa'] = float(Q[0] @ K[0])
NUM['q_pmy'] = (torch.tensor([0.2, 0.5, 0.3]) - torch.tensor([0., 1., 0.])).tolist()
NUM['ctx_cells'] = {str(T_): T_ * T_ for T_ in [6, 1024, 2048, 8192]}
torch.manual_seed(123)
NUM['q_block_params'] = sum(p_.numel() for p_ in Block(16, 2, 64).parameters())
assert NUM['q_block_params'] == 3216 and NUM['q_masked_T4'] == 6
assert abs(NUM['q_S_aa'] - S[0, 0].item()) < 1e-12

## Final check
Literal values quoted in the PDF, then a full comparison with `src/numbers.json`.

In [41]:
# Key numbers quoted in the guide (literal values)
assert abs(NUM['final_loss'] - 0.000534) < 1e-06, 'final_loss'
assert abs(NUM['loss0'] - 2.2412) < 0.0001, 'loss0'
assert abs(NUM['loss_hand'] - 1.9558) < 0.0001, 'loss_hand'
assert abs(NUM['n_params'] - 3413) < 0.5, 'n_params'
assert NUM['greedy'] == 'a bad monkey is still a monkey'
assert abs(NUM['A_causal'][1][0] - 0.8449) < 1e-4
assert abs(NUM['ablate']['self_only_no_pe']['loss'] - 0.2316) < 1e-4
print('all literal checks passed')

all literal checks passed


In [42]:
import json
ref = json.load(open('src/numbers.json'))
def same(a, b, path=''):
    if isinstance(a, dict):
        assert set(a) == set(b), path
        for k in a: same(a[k], b[k], path + '/' + k)
    elif isinstance(a, (list, tuple)):
        assert len(a) == len(b), path
        for i, (u, v) in enumerate(zip(a, b)): same(u, v, f'{path}[{i}]')
    elif isinstance(a, float):
        assert abs(a - b) < 1e-9 * max(1, abs(a)), (path, a, b)
    else:
        assert a == b, (path, a, b)
same(json.loads(json.dumps(NUM)), ref)
print('every number in numbers.json reproduced:', len(ref), 'top-level entries')

every number in numbers.json reproduced: 94 top-level entries
